# Koebuta Notebook v2 — 生成専用試作
GPUランタイムで上から実行してください。シナリオ編集はHTML版で行います。
Driveは任意です。利用する場合は下のセルでマウントしてください。
Colabの利用条件・モデルと音声素材の利用条件はv1と同様に確認してください。


## 1. セットアップ（GPU確認・Irodori導入・モデルの取得）

既定ではIrodori-TTSの公開リポジトリから `main` を取得し、実際のコミットIDを表示します。再現性が必要な場合は `KOEBUTA_IRODORI_COMMIT` を指定してください。
依存関係の導入後にランタイム再起動が必要になった場合は、再起動後にセルを順番に再実行してください。**再実行前に未保存の成果物がないか確認してください。**


In [ ]:
# Driveを利用する場合だけTrueにして実行
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import subprocess, sys, os, shutil, re
from pathlib import Path

# 指定したい場合のみ環境変数でコミットを固定。未指定なら最新版mainを使用し、実際のコミットを記録します。
# 既存の旧コミットはv4.1との互換性を確認できなかったため、ここでは固定しません。
IRODORI_COMMIT = os.environ.get('KOEBUTA_IRODORI_COMMIT','').strip()
IRODORI_DIR = Path('/content/Irodori-TTS')
MODEL_ID = 'Aratako/Irodori-TTS-v4.1-Small'
MODEL_REVISION = '2b28324dc263ed5e6638b3cf3dd94c82ead07b4b'
CODEC_ID = 'Aratako/Semantic-DACVAE-Japanese-32dim'
CODEC_REVISION = '47376ee24834d7a05a48ebabfe3cde29b3c5e214'

# GPU確認
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
print(r.stdout.strip() or 'GPUが見えません。ランタイム > ランタイムのタイプを変更 で GPU を選んでください。')
assert r.returncode == 0, 'GPUランタイムで実行してください'

# 既存のフォルダーは削除せず、別名に退避する必要がある場合は作業を中止
if IRODORI_DIR.exists() and not (IRODORI_DIR / '.git').is_dir():
    raise RuntimeError(f'{IRODORI_DIR} はGit管理外です。手動で確認してください（自動削除しません）')
if not IRODORI_DIR.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', str(IRODORI_DIR)])
# 利用者の変更がある既存リポジトリを上書きしない
status = subprocess.check_output(['git','-C',str(IRODORI_DIR),'status','--porcelain'],text=True).strip()
if status:
    raise RuntimeError('既存のIrodori-TTSに未コミット変更があります。変更を保護するため停止します')
target = IRODORI_COMMIT or 'main'
subprocess.check_call(['git','-C',str(IRODORI_DIR),'fetch','--depth','1','origin',target])
subprocess.check_call(['git','-C',str(IRODORI_DIR),'checkout','--detach','FETCH_HEAD'])

IRODORI_RESOLVED_COMMIT = subprocess.check_output(['git','-C',str(IRODORI_DIR),'rev-parse','HEAD'],text=True).strip()
print('Irodori-TTS:', IRODORI_RESOLVED_COMMIT)

# 取得した上流ファイルは変更せず、Python 3.13ではインストール用のコピーにだけ
# sentencepieceの互換範囲を設定する。再実行してもGit作業ツリーは汚れない。
req_file = IRODORI_DIR / 'requirements.txt'
req_copy = Path('/content/koebuta_requirements_runtime.txt')
req_text = req_file.read_text(encoding='utf-8')
if sys.version_info >= (3,13):
    req_text, changed = re.subn(r'(?m)^sentencepiece[^\n]*$', 'sentencepiece>=0.2.0', req_text)
    if changed != 1:
        print('注意: sentencepiece依存の記述形式が変わったため、Python 3.13互換調整を適用していません')
req_copy.write_text(req_text,encoding='utf-8')

# 依存を導入
print("パッケージのインストールを開始します...")

print("1. requirements.txt から依存ライブラリをインストール中...")
res_req = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_copy)], capture_output=True, text=True)
if res_req.returncode != 0:
    print("=== requirements.txt インストールエラー ===")
    print(res_req.stdout)
    print(res_req.stderr)
    raise RuntimeError("requirements.txt のインストールに失敗しました")

print("2. Irodori-TTS 本体をインストール中...")
res_main = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(IRODORI_DIR)], capture_output=True, text=True)
if res_main.returncode != 0:
    print("=== 本体インストールエラー ===")
    print(res_main.stdout)
    print(res_main.stderr)
    raise RuntimeError("本体のインストールに失敗しました")

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==6.5.1', 'regex>=2024.11.6'])

# モデルとコーデックを取得
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(MODEL_ID, 'model.safetensors', revision=MODEL_REVISION)
codec = hf_hub_download(CODEC_ID, 'weights.pth', revision=CODEC_REVISION)
print('checkpoint:', ckpt)
print('codec:', codec)
print('セットアップ完了。')


## 2. 本体ロジック（検証・読み変換・分割）

画面から使う関数群です。実行するだけで構いません。
元版 `irodori_cli` の規則（形式・感情表・ルビ変換・分割）をもとに、ノートブック用に単純化してあります。

In [ ]:
import copy, hashlib, json, math, random, re, shutil, unicodedata
from pathlib import Path

WORK = Path('/content/koebuta_work')
SEG_DIR = WORK / 'segments'      # 生成済み発話WAVとキー
VOICE_DIR = WORK / 'voices'      # 採用した参照WAV
OUT_DIR = WORK / 'out'
for d in (SEG_DIR, VOICE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 感情ID -> 先頭に付ける絵文字（元版 core.EMOTIONS と同じ対応）
EMOTIONS = dict(zip(
    ['neutral','joy','anger','sad','kind','tease','anxiety','tension','surprise','tired','whisper','giggle','fear','sleepy','resolve'],
    ['','😆','😠','😭','🫶','😏','🥺','😰','😲','😮‍💨','👂','🤭','😱','😪','💥']))
RUBY_FORMATS = ('none', 'aozora', 'narou', 'kakuyomu', 'pixiv')

# Web UIではGradioのセッションごとに独立した状態を使う。
# GPUのモデルだけは共有して構わないので、ランタイムは別途グローバルで管理する。
from contextvars import ContextVar
from collections.abc import MutableMapping

def new_session():
    return {'project':None, 'voices':{}, 'sample':None, 'last_files':[],
            'undo':[], 'plan_warnings':[], 'length_min':12, 'length_max':140}

_CTX = ContextVar('koebuta_session',default=None)
_FALLBACK_SESSION = new_session()  # ノートブック内からの直接呼び出し用

class SessionData(MutableMapping):
    def _data(self): return _CTX.get() if _CTX.get() is not None else _FALLBACK_SESSION
    def __getitem__(self,k): return self._data()[k]
    def __setitem__(self,k,v): self._data()[k]=v
    def __delitem__(self,k): del self._data()[k]
    def __iter__(self): return iter(self._data())
    def __len__(self): return len(self._data())

STATE = SessionData()
_RUNTIME = {'instance':None, 'model':None}
# voices: プロファイルキー -> 参照WAVパス。単独方式は 'shared'、複数方式は role の id をキーにする。

# ---------- 標準シナリオJSONの寛容な読み込み ----------
SCENARIO_KEYS = {'schema_version', 'source', 'speakers', 'segments'}
SPEAKER_KEYS = {'name', 'kind', 'bio', 'voice_caption', 'acting_description', 'age', 'gender', 'pitch', 'timbre', 'habitual_style'}
SEGMENT_KEYS = {'id', 'speaker_id', 'text', 'emotion', 'acting_description', 'tts_text',
                'pause_after_ms', 'segment_type', 'text_format', 'blank_lines_after'}
ATTR_VALUES = {
    'age': {'young_adult','adult','elderly'}, 'gender': {'feminine','masculine'},
    'pitch': {'high','mid','low'},
    'timbre': {'clear','warm','soft','husky','bright','resonant'},
    'habitual_style': {'gentle','measured','brisk','formal','casual','calm'},
}

def _valid_id(v):
    return isinstance(v, str) and bool(v.strip())

def _optional_string(v, label, warnings):
    if v is None: return ''
    if isinstance(v, str): return v
    warnings.append(f'{label} は文字列ではないため、指定なしとして扱いました')
    return ''

def _nonnegative_int(v):
    return type(v) is int and v >= 0

def _temp_id(used, i):
    candidate = f'_pending_{i}'
    while candidate in used:
        candidate += '_'
    used.add(candidate)
    return candidate

def _safe_scalar(v, default, predicate, label, warnings):
    if predicate(v): return v
    warnings.append(f'{label} が不正なため標準設定を使用しました')
    return default


# ---------- Scenario Inspector v0.1.6 P0 compatibility ----------
from pathlib import PurePosixPath

_P0_BOOL_KEYS = ('use_emotion', 'use_acting', 'narrator_emotion')
_P0_INT_RANGES = {
    'seed': (0, 2147483647),
    'steps': (1, 200),
    'default_pause_ms': (0, 60000),
    'default_blank_line_pause_ms': (0, 60000),
}

def checked_relative_wav(value):
    """Verify portable project-relative WAV spelling before filesystem access."""
    if not isinstance(value, str) or not value or value.strip() != value:
        raise ValueError('WAVの相対パスが空または不正です')
    if '\\' in value or '\x00' in value or ':' in value or value.startswith('/'):
        raise ValueError('WAVには作品フォルダ相対の / 区切りパスを指定してください')
    parts = value.split('/')
    if any(part in ('', '.', '..') for part in parts) or PurePosixPath(value).suffix.lower() != '.wav':
        raise ValueError('WAVパスに空要素、親ディレクトリ参照、または不正な拡張子があります')
    return value

def notebook_settings(project):
    """Read optional Inspector settings. Missing keys retain Notebook defaults."""
    raw = project.get('_extras', {}).get('koebuta_settings')
    if raw is None:
        return {}
    if not isinstance(raw, dict):
        raise ValueError('koebuta_settings はオブジェクトにしてください')
    for key in _P0_BOOL_KEYS:
        if key in raw and type(raw[key]) is not bool:
            raise ValueError(f'koebuta_settings.{key} は真偽値で指定してください')
    for key, (low, high) in _P0_INT_RANGES.items():
        if key in raw and (type(raw[key]) is not int or not low <= raw[key] <= high):
            raise ValueError(f'koebuta_settings.{key} は {low}〜{high} の整数で指定してください')
    if 'reading_mode' in raw and raw['reading_mode'] not in ('shared', 'cast'):
        raise ValueError('koebuta_settings.reading_mode は shared または cast にしてください')
    if 'default_text_format' in raw and raw['default_text_format'] not in RUBY_FORMATS:
        raise ValueError('koebuta_settings.default_text_format が不正です')
    if raw.get('shared_reference_wav') is not None:
        checked_relative_wav(raw['shared_reference_wav'])
    return raw

def check_tts_parts(segment):
    """Validate the opt-in TTS boundaries without ever rewriting original text."""
    extra = segment.get('_extras', {})
    if 'koebuta_tts_parts' not in extra:
        return None
    parts = extra['koebuta_tts_parts']
    if (not isinstance(parts, list) or len(parts) < 2
            or any(not isinstance(part, str) or not part.strip() for part in parts)):
        raise ValueError('koebuta_tts_parts は2件以上の空でない文字列配列にしてください')
    if not isinstance(segment.get('tts_text'), str) or ''.join(parts) != segment['tts_text']:
        raise ValueError('koebuta_tts_parts の連結結果が tts_text と一致しません')
    return parts

def normalize_scenario(raw):
    '''受け付けた値は失わず編集用に保持。音声生成の可否はvalidate_projectで判断する。'''
    if not isinstance(raw, dict):
        raise ValueError('JSONの最上位はオブジェクトにしてください')
    legacy = raw.get('version') == 1 and 'roles' in raw and 'speakers' not in raw
    if not legacy:
        version = raw.get('schema_version')
        if version not in (None, 1) or type(version) not in (int, type(None)):
            raise ValueError(f'未対応のシナリオ形式バージョン: {version!r}')
        if version is None and not isinstance(raw.get('speakers'), dict):
            raise ValueError('形式バージョンがなく、v1のspeakersを確認できません')

    warnings = []
    if not legacy and 'schema_version' not in raw:
        warnings.append('形式バージョンがないためv1形式として読み込みました')
    roles = []
    if legacy:
        source_roles = raw.get('roles')
        if not isinstance(source_roles, list):
            warnings.append('話者一覧が不正なため、空の一覧として読み込みました')
            source_roles = []
        role_items = [(r.get('id') if isinstance(r, dict) else None, r) for r in source_roles]
    else:
        source_roles = raw.get('speakers')
        if not isinstance(source_roles, dict):
            warnings.append('話者一覧が見つからないか形式が不正です')
            source_roles = {}
        role_items = list(source_roles.items())
    used_roles = set()
    for i, (key, p) in enumerate(role_items, 1):
        if not isinstance(p, dict):
            warnings.append(f'話者#{i} の設定がオブジェクトではありません')
            p = {}
        if not _valid_id(key) or key in used_roles:
            warnings.append(f'話者#{i} のIDが欠落または重複しています。確認が必要です')
            ident = _temp_id(used_roles, i)
            pending = True
        else:
            ident = key
            used_roles.add(ident)
            pending = False
        role = {'id': ident, 'name': p.get('name') if isinstance(p.get('name'), str) and p['name'].strip() else ident,
                'kind': p.get('kind') if p.get('kind') in ('narrator','character') else None,
                'bio': '', 'caption': '', 'acting': '', '_extras': {}, '_pending_id': pending}
        if role['name'] == ident and not (isinstance(p.get('name'), str) and p['name'].strip()):
            warnings.append(f'話者 {ident}: 表示名がないため話者IDを使います')
        if role['kind'] is None:
            warnings.append(f'話者 {ident}: 種別が未確定です（地の文か登場人物かを確認してください）')
        names = [('bio','bio'), ('voice_caption','caption'), ('acting_description','acting')]
        for source_key, out_key in names:
            raw_value = p.get(source_key) if not legacy else p.get({'caption':'caption','acting':'acting'}.get(out_key, source_key))
            role[out_key] = _optional_string(raw_value, f'話者 {ident} / {source_key}', warnings)
        for attr, values in ATTR_VALUES.items():
            val = p.get(attr)
            if val in values if isinstance(val, str) else False:
                role[attr] = val
            elif val is not None:
                warnings.append(f'話者 {ident}: {attr} が不正なため無視しました')
        role['_extras'] = {k: copy.deepcopy(v) for k,v in p.items() if k not in (SPEAKER_KEYS if not legacy else SPEAKER_KEYS | {'id','caption','acting'})}
        roles.append(role)

    rows = raw.get('segments')
    if not isinstance(rows, list):
        warnings.append('台本項目が見つからないか配列ではありません')
        rows = []
    used_segments = set()
    segments = []
    for i, s in enumerate(rows, 1):
        if not isinstance(s, dict):
            warnings.append(f'台本項目#{i} がオブジェクトではありません')
            s = {}
        original_id = s.get('id')
        if not _valid_id(original_id) or original_id in used_segments:
            sid = _temp_id(used_segments, i)
            pending = True
            warnings.append(f'台本項目#{i}: IDの欠落・重複を検出。仮ID {sid} を割り当てました')
        else:
            sid = original_id
            used_segments.add(sid)
            pending = False
        segment_type = s.get('segment_type', 'speech')
        if segment_type not in ('speech', 'separator') if isinstance(segment_type, str) else True:
            warnings.append(f'{sid}: 不明な項目種別です。確認するまで読み上げません')
        emotion = s.get('emotion', 'neutral')
        if not isinstance(emotion,str) or emotion not in EMOTIONS:
            warnings.append(f'{sid}: 未知の感情を「平静」に置き換えました')
            emotion = 'neutral'
        raw_role = s.get('role') if legacy else s.get('speaker_id')
        seg = {'id': sid, 'role': raw_role if isinstance(raw_role, str) else None,
               'text': s.get('text'), 'emotion': emotion, 'acting':
               _optional_string(s.get('acting' if legacy else 'acting_description'), f'{sid} / 演技指示', warnings),
               'segment_type': segment_type, '_pending_id': pending, '_extras': {}}
        if not isinstance(seg['text'], str) and segment_type == 'speech':
            warnings.append(f'{sid}: 原文が欠落または文字列ではありません。補完せず保持しています')
        for k in ('tts_text', 'pause_after_ms', 'text_format', 'blank_lines_after'):
            if k not in s: continue
            val = s[k]
            if k == 'tts_text':
                if isinstance(val, str) and val.strip(): seg[k] = val
                else: warnings.append(f'{sid}: 無効な読み上げ文は無視しました')
            elif k in ('pause_after_ms','blank_lines_after'):
                if _nonnegative_int(val): seg[k] = val
                else: warnings.append(f'{sid}: {k} の異常値は無視しました')
            elif k == 'text_format':
                if isinstance(val,str) and val in RUBY_FORMATS: seg[k] = val
                else: warnings.append(f'{sid}: 不明なルビ形式は無視しました')
        seg['_extras'] = {k: copy.deepcopy(v) for k,v in s.items() if k not in (SEGMENT_KEYS if not legacy else SEGMENT_KEYS | {'role','acting'})}
        segments.append(seg)

    source = raw.get('source') if isinstance(raw.get('source'), str) else None
    if 'source' in raw and source is None: warnings.append('原稿全文が文字列ではないため、原文照合を省略しました')
    reading_mode = raw.get('reading', 'cast') if legacy else 'cast'
    if reading_mode not in ('shared','cast'): reading_mode = 'cast'
    project = {
        'title': raw.get('title', '新しい朗読') if legacy else '新しい朗読',
        'source': source, 'roles': roles, 'segments': segments, 'reading': reading_mode,
        'baseCaption': raw.get('baseCaption','自然で聞き取りやすい日本語の朗読。') if legacy else '自然で聞き取りやすい日本語の朗読。',
        'seed': raw.get('seed',314) if legacy else 314,
        'steps': raw.get('steps',40) if legacy else 40,
        'pause': raw.get('pause',0.28) if legacy else 0.28,
        'acting': raw.get('acting',False) if legacy else False,
        'emotion': raw.get('emotion',True) if legacy else True,
        'narratorEmotion': raw.get('narratorEmotion',False) if legacy else False,
        'textOptions': raw.get('textOptions',{}) if legacy and isinstance(raw.get('textOptions',{}),dict) else {},
        '_load_warnings': warnings,
        '_extras': {k:copy.deepcopy(v) for k,v in raw.items() if k not in (SCENARIO_KEYS if not legacy else {'version','source','roles','segments','title','reading','baseCaption','seed','steps','pause','acting','emotion','narratorEmotion','textOptions'})},
        '_legacy': legacy,
    }
    project['steps'] = _safe_scalar(project['steps'],40,lambda v:type(v) is int and v in (20,40,60),'生成ステップ数',warnings)
    project['seed'] = _safe_scalar(project['seed'],314,lambda v:type(v) is int and 0 <= v <= 2147483647,'生成シード',warnings)
    project['pause'] = _safe_scalar(project['pause'],0.28,lambda v:type(v) in (int,float) and math.isfinite(v) and 0 <= v <= 30,'標準の間',warnings)
    # JSONがreading_modeを指定するときのみNotebook内部へ反映。
    # 未指定の旧JSONでは既存のUI選択値を優先する。
    try:
        cfg = notebook_settings(project)
        if 'reading_mode' in cfg:
            project['reading'] = cfg['reading_mode']
    except ValueError:
        pass  # 壊れた入力は編集用に保持し、validate_projectで停止する。
    return project

def validate_project(project):
    '''現在の状態を動的検証する。問題はユーザーに知らせ、データは破棄しない。'''
    issues = []
    def add(level, item, message): issues.append((level, item, message))
    if not project: return [('エラー','全体','シナリオがありません')]
    try:
        cfg = notebook_settings(project)
    except ValueError as exc:
        add('エラー', '生成設定', str(exc))
        cfg = {}
    for r in project.get('roles', []):
        path = r.get('_extras', {}).get('reference_wav')
        if path is not None:
            try:
                checked_relative_wav(path)
            except ValueError as exc:
                add('エラー', f'話者 {r.get("id")}', str(exc))
    role_ids = set()
    for r in project.get('roles',[]):
        rid = r.get('id')
        if not _valid_id(rid) or rid in role_ids or r.get('_pending_id'):
            add('要確認',f'話者 {rid}','話者IDを確定してください')
        role_ids.add(rid)
        if r.get('kind') not in ('narrator','character'):
            add('要確認',f'話者 {rid}','地の文か登場人物かを確認してください')
    if not project.get('segments'):
        add('エラー','台本','台本項目がありません')
    used = set()
    for s in project.get('segments',[]):
        sid = s.get('id')
        if not _valid_id(sid) or sid in used or s.get('_pending_id'):
            add('要確認',f'項目 {sid}','発話IDを確定してください')
        used.add(sid)
        typ = s.get('segment_type','speech')
        if typ not in ('speech','separator'):
            add('要確認',f'項目 {sid}',f'不明な項目種別: {typ!r}')
            continue
        if typ == 'separator': continue
        if not isinstance(s.get('text'),str) or not s['text'].strip():
            add('エラー',f'発話 {sid}','原文が欠落または空です')
        if s.get('role') not in role_ids:
            add('要確認',f'発話 {sid}',f'未登録の話者ID: {s.get("role")!r}')
        try:
            check_tts_parts(s)
        except ValueError as exc:
            add('エラー', f'発話 {sid}', str(exc))
        blank_ms = s.get('_extras', {}).get('blank_line_pause_ms')
        if blank_ms is not None and (type(blank_ms) is not int or not 0 <= blank_ms <= 60000):
            add('エラー', f'発話 {sid}', 'blank_line_pause_ms は0〜60000の整数にしてください')
    # 入力内の完全性の証明は行わない。検出可能な順序・部分一致のみ検査する。
    source = project.get('source')
    if isinstance(source,str):
        pos = 0
        for s in project.get('segments',[]):
            original = s.get('text')
            if isinstance(original,str) and original:
                idx = source.find(original,pos)
                if idx < 0:
                    add('要確認',f'項目 {s.get("id")}', '原稿全文中に同じ原文を順序どおり見つけられません（部分照合）')
                    break
                pos = idx + len(original)
    return issues

def issues_message(project, preview=12):
    issues = validate_project(project)
    warns = project.get('_load_warnings',[]) if project else []
    lines = [f'要確認・エラー: {len(issues)}件 / 読込時の注意: {len(warns)}件']
    lines += [f'⚠ {level} | {item} | {msg}' for level,item,msg in issues[:preview]]
    lines += [f'ℹ {s}' for s in warns[:max(0,preview-len(issues))]]
    if len(issues)+len(warns)>preview: lines.append(f'ほか {len(issues)+len(warns)-preview} 件')
    if not isinstance(project.get('source') if project else None,str):
        lines.append('原稿全文がないため原文照合は未実施です')
    else:
        lines.append('原文は原稿全文との順序・部分一致のみ照合しました。完全性を保証するものではありません')
    return '\n'.join(lines)

def require_ready(project):
    issues = validate_project(project)
    if issues:
        examples = '\n'.join(f'{item}: {msg}' for _,item,msg in issues[:8])
        raise ValueError(f'生成前に{len(issues)}件の項目を確認してください:\n{examples}')

def export_scenario(project):
    if not project: raise ValueError('シナリオがありません')
    # IDは参照関係の核となるため、仮IDのまま標準形式として出力しない。
    for r in project['roles']:
        if r.get('_pending_id'): raise ValueError(f'話者 {r["id"]}: 仮IDを確定してから保存してください')
    for s in project['segments']:
        if s.get('_pending_id'): raise ValueError(f'発話 {s["id"]}: 仮IDを確定してから保存してください')
    known={r['id'] for r in project['roles']}
    for segment in project['segments']:
        if segment.get('segment_type','speech')=='speech' and segment.get('role') not in known:
            raise ValueError(f'未登録の話者を参照しています: {segment.get("role")}')
    out = copy.deepcopy(project.get('_extras',{}))
    out.update({'schema_version':1,'speakers':{},'segments':[]})
    if project.get('source') is not None: out['source'] = project['source']
    else: out.pop('source',None)
    for r in project['roles']:
        ident = r['id']
        d = copy.deepcopy(r.get('_extras',{}))
        d.update({'name':r['name']})
        if r.get('kind') in ('narrator','character'): d['kind'] = r['kind']
        for src,dst in [('bio','bio'),('caption','voice_caption'),('acting','acting_description')]:
            if r.get(src): d[dst] = r[src]
            else: d.pop(dst,None)
        for attr in ATTR_VALUES:
            if attr in r: d[attr] = r[attr]
            else: d.pop(attr,None)
        out['speakers'][ident] = d
    for s in project['segments']:
        d = copy.deepcopy(s.get('_extras',{}))
        d['id'] = s['id']
        if s.get('role') is not None: d['speaker_id'] = s['role']
        else: d.pop('speaker_id',None)
        if s.get('text') is not None: d['text'] = s['text']
        else: d.pop('text',None)
        d['emotion'] = s.get('emotion','neutral')
        if s.get('acting'): d['acting_description'] = s['acting']
        for k in ('tts_text','pause_after_ms','segment_type','text_format','blank_lines_after'):
            if k in s: d[k] = s[k]
            else: d.pop(k,None)
        out['segments'].append(d)
    return out


def is_narrator(role):
    return role.get('kind') == 'narrator'

# ---------- 決定的ルビ変換（元版 text_conversion.reading と同じ規則・結果） ----------
_KANJI = r'\u3400-\u9fff\U00020000-\U0002ffff々〆〇ヶ'
def reading(body, fmt):
    '''本文を読み上げ用テキストへ変換し、(読み, 警告リスト) を返す。'''
    if fmt not in RUBY_FORMATS: raise ValueError('未対応のルビ形式です')
    if fmt == 'none': return body, []
    warnings, text = [], body
    def sub(pattern, repl, s):
        return re.sub(pattern, repl, s)
    if fmt == 'pixiv':
        text = sub(r'\[\[rb:([^\[\]\n>]+?)\s*>\s*([^\[\]\n]+?)\]\]', lambda m: m[2].strip(), text)
        if '[[' in text or ']]' in text: warnings.append('未対応または不正なpixiv記法があります')
        return text, warnings
    if fmt == 'aozora':
        text = sub(r'※［＃小書き平仮名([ぁ-ゖ])、[0-9０-９]+[-－][0-9０-９]+］', lambda m: m[1], text)
        known = (r'(?:改ページ|改丁|改見開き|ここから[０-９0-9]+字下げ|ここで字下げ終わり|[０-９0-9]+字下げ|'
                 r'「[^「」［］\n]+」に傍点|「[^「」［］\n]+」は[大中小]見出し|[大中小]見出し(?:終わり)?)')
        text = sub(r'［＃([^［］\n]*)］', lambda m: '' if re.fullmatch(known, m[1]) else m[0], text)
    marker = '\ue000'
    while marker in text: marker += '\ue000'
    if fmt == 'kakuyomu': text = text.replace('｜《', marker + 'a')
    if fmt == 'narou':
        text = sub(r'[｜|]([（(])', lambda m: marker + ('b' if m[1] == '(' else 'c'), text)
    if fmt == 'kakuyomu': text = sub(r'《《([^《》\n]+)》》', r'\1', text)
    bars = '｜' if fmt == 'aozora' else '｜|'
    text = sub('[' + bars + r']([^｜|《》\n]+)《([^《》\n]+)》', lambda m: m[2], text)
    text = sub('[' + _KANJI + r']+《([^《》\n]+)》', lambda m: m[1], text)
    if fmt == 'narou':
        text = sub('[' + _KANJI + r']+\(([ぁ-ゖァ-ヺー]+)\)', lambda m: m[1], text)
        text = sub('[' + _KANJI + r']+（([ぁ-ゖァ-ヺー]+)）', lambda m: m[1], text)
    if any(mk in text for mk in ('《', '》', '［＃', '｜', '］')): warnings.append('未対応または不正なルビ・注記があります')
    for suffix, value in (('a', '《'), ('b', '('), ('c', '（')):
        text = text.replace(marker + suffix, value)
    if not text.strip(): warnings.append('変換後に読み上げる本文がありません')
    return text, warnings

# ---------- 音声生成用の自然な発話分割（原文の文字列は厳密に保持） ----------
def split_text(text, limit=140):
    """v1.0.1型の控えめな分割。原文を必ず保持し、短い末尾を孤立させない。

    生成時のみ使用。台本JSONの区切りは変えず、モデル入力の安全のため
    異常に長い部分のみ読点・空白を優先して分割する。句読点がなく長い場合は
    文字数での分割が起き得るため、UIで長文警告・手動調整を推奨する。
    """
    if not isinstance(text,str) or not text.strip():
        raise ValueError('分割する本文が空です')
    if not isinstance(limit,int) or limit<20:
        raise ValueError('分割目安は20文字以上の整数にしてください')
    units = re.findall(r'[^\n。！？!?]*[。！？!?][」』”"）)]*|[^\n。！？!?]*\n+|[^\n。！？!?]+', text)
    if ''.join(units)!=text: raise ValueError('発話分割で本文を保持できませんでした')
    parts,current=[],''
    for unit in units:
        if current and (len(current)+len(unit)>limit or current.endswith('\n')):
            parts.append(current);current=''
        while len(unit)>limit:
            if current:parts.append(current);current=''
            pos=[m.end() for m in re.finditer(r'[、，,；;：:\s]+',unit[:limit])
                 if m.end()>=limit//3 and unit[:m.end()].strip()]
            cut=pos[-1] if pos else limit
            # 数文字だけ余る場合は不自然な切断をせず、長い1単位として扱う。
            if len(unit)-cut < max(12,limit//10) and len(unit)<=int(limit*1.5):
                break
            import regex
            boundaries=[m.end() for m in regex.finditer(r'\X',unit)]
            before=[k for k in boundaries if k<=cut]
            cut=before[-1] if before else boundaries[0]
            if cut<=0 or cut>=len(unit):break
            parts.append(unit[:cut]);unit=unit[cut:]
        current+=unit
    if current:parts.append(current)
    clean,pending=[],''
    for part in parts:
        if not part.strip() and clean:clean[-1]+=part
        elif not part.strip():pending+=part
        else:clean.append(pending+part);pending=''
    if ''.join(clean)!=text or any(not p.strip() for p in clean):
        raise ValueError('空の分割発話があります')
    return clean

print('本体ロジックを読み込みました')


## 3. 合成ロジック（計画・生成・連結）

声の割り当て、生成計画（読み・演技・感情・間の解決）、モデル読み込み、生成と連結です。

In [ ]:
import numpy as np
import soundfile as sf

DEFAULT_PAUSE_MS = 280
SEPARATOR_PAUSE_MS = 1200

def voice_key(project, role):
    return 'shared' if project.get('reading', 'shared') == 'shared' else role['id']

def resolve_pauses(project, global_pause_ms, default_blank_line_pause_ms=1200):
    """Explicit pause > separator/blank-line pause > global pause; 0 is explicit."""
    if type(global_pause_ms) is not int or not 0 <= global_pause_ms <= 60000:
        raise ValueError('共通休止時間が不正です')
    if type(default_blank_line_pause_ms) is not int or not 0 <= default_blank_line_pause_ms <= 60000:
        raise ValueError('空行時の共通休止時間が不正です')
    result, pending_sep, previous = {}, None, None
    for s in project['segments']:
        if s.get('segment_type') == 'separator':
            v = s.get('pause_after_ms', SEPARATOR_PAUSE_MS)
            pending_sep = max(v, pending_sep if pending_sep is not None else 0)
            continue
        if previous is not None and pending_sep is not None and 'pause_after_ms' not in previous:
            result[previous['id']] = max(result[previous['id']], pending_sep)
        pending_sep = None
        if 'pause_after_ms' in s:
            result[s['id']] = s['pause_after_ms']
        elif s.get('blank_lines_after', 0) > 0:
            blank = s.get('_extras', {}).get('blank_line_pause_ms')
            result[s['id']] = (blank if blank is not None
                               else max(global_pause_ms, default_blank_line_pause_ms))
        else:
            result[s['id']] = global_pause_ms
        previous = s
    return result

def build_plan(project, voices, *, use_acting, use_emotion, narrator_emotion, seed, steps, global_pause_ms):
    """Build generation units while respecting Inspector's optional TTS boundaries."""
    require_ready(project)
    cfg = notebook_settings(project)
    roles = {r['id']: r for r in project['roles']}
    default_fmt = cfg.get('default_text_format', (project.get('textOptions') or {}).get('format', 'none'))
    pauses = resolve_pauses(project, global_pause_ms, cfg.get('default_blank_line_pause_ms', 1200))
    used_role_ids = {s['role'] for s in project['segments'] if s.get('segment_type', 'speech') == 'speech'}
    missing = []
    if project.get('reading', 'shared') == 'shared':
        if 'shared' not in voices:
            missing.append('共用の声')
    else:
        for rid in sorted(used_role_ids):
            if rid not in voices:
                missing.append(roles[rid]['name'])
    if missing:
        raise ValueError('参照音声が未設定です: ' + '、'.join(missing) + '（作品フォルダ内のWAVまたは手動割り当てを確認してください）')
    plan, blocked, used_unit_ids = [], [], set()
    STATE['plan_warnings'] = []
    for s in project['segments']:
        if s.get('segment_type') == 'separator':
            continue
        role = roles[s['role']]
        fmt = s.get('text_format', default_fmt)
        if isinstance(s.get('tts_text'), str) and s['tts_text'].strip():
            spoken, warns = s['tts_text'], []
        else:
            spoken, warns = reading(s['text'], fmt)
        if warns:
            blocked.append(f"{s['id']}: {' / '.join(warns)}")
        prescribed = check_tts_parts(s)
        if prescribed is None:
            pieces = split_text(spoken, 140)
        else:
            pieces = []
            for n, part in enumerate(prescribed, 1):
                if len(part) > 140:
                    safe_subparts = split_text(part, 140)
                    pieces.extend(safe_subparts)
                    STATE['plan_warnings'].append(
                        f"{s['id']} のTTS片#{n}は140文字超のためモデル入力上の安全分割を追加しました")
                else:
                    pieces.append(part)
            if ''.join(pieces) != spoken:
                raise ValueError(f"{s['id']}: TTS分割により本文が変化しました")
        for piece in pieces:
            if len(piece.strip()) < 10:
                STATE['plan_warnings'].append(
                    f'{s["id"]}: {len(piece.strip())}文字「{piece.strip()}」— 短い生成片です')
        for i, piece in enumerate(pieces):
            text = piece
            if i == 0 and use_emotion and role.get('kind') in ('narrator', 'character') and (
                    narrator_emotion or not is_narrator(role)):
                text = EMOTIONS[s['emotion']] + text
            caption = project.get('baseCaption', '')
            if use_acting:
                caption = '\n'.join(p for p in [
                    caption.rstrip(), role.get('acting', '').strip(), s.get('acting', '').strip()] if p)
            unit_id = s['id'] if len(pieces) == 1 else f"{s['id']}#{i + 1}"
            if unit_id in used_unit_ids:
                raise ValueError(f'生成単位IDが重複しています: {unit_id}')
            used_unit_ids.add(unit_id)
            ref = voices[voice_key(project, role)]
            plan.append({
                'unit_id': unit_id, 'segment_id': s['id'],
                'text': text, 'caption': caption or None, 'ref_wav': str(ref),
                'seed': seed, 'num_steps': steps,
                'pause_after_ms': pauses.get(s['id'], global_pause_ms) if i == len(pieces) - 1 else 0})
    if blocked:
        raise ValueError('読み変換で確認が必要な発話があります:\n' + '\n'.join(blocked))
    if not plan:
        raise ValueError('読み上げる本文がありません')
    return plan

def resolve_project_wavs(project, folder):
    """Safely resolve Inspector WAVs from the folder holding scenario.json.

    Returns (available voices, issues). UI-adopted voices can override these paths.
    """
    root = Path(folder).expanduser().resolve(strict=True)
    if not root.is_dir():
        raise ValueError('作品フォルダが存在しません')
    cfg = notebook_settings(project)
    roles = {r['id']: r for r in project['roles']}
    used = {s['role'] for s in project['segments']
            if s.get('segment_type', 'speech') == 'speech'}
    references = (
        {'shared': cfg.get('shared_reference_wav')} if project.get('reading') == 'shared'
        else {rid: roles[rid].get('_extras', {}).get('reference_wav') for rid in used if rid in roles}
    )
    voices, issues = {}, []
    for key, relative in references.items():
        if relative is None:
            issues.append(f'{key}: 参照WAVがJSONに指定されていません（UIで手動指定できます）')
            continue
        try:
            checked_relative_wav(relative)
            path = root.joinpath(*relative.split('/')).resolve(strict=True)
            if not path.is_relative_to(root) or not path.is_file():
                raise ValueError('作品フォルダの外を参照しています')
            info = sf.info(str(path))
            if info.frames <= 0 or info.channels <= 0:
                raise ValueError('WAVが空です')
            voices[key] = str(path)
        except (OSError, RuntimeError, ValueError) as exc:
            issues.append(f'{key}: {relative}: {exc}')
    return voices, issues

def request_key(unit, model_id):
    h = hashlib.sha256()
    h.update(Path(unit['ref_wav']).read_bytes())
    ref_hash = h.hexdigest()
    return hashlib.sha256(json.dumps({'model': model_id, 'model_revision':MODEL_REVISION, 'codec':CODEC_ID, 'codec_revision':CODEC_REVISION, 'upstream':globals().get('IRODORI_RESOLVED_COMMIT',globals().get('IRODORI_COMMIT','unknown')), 'ref': ref_hash, **{k: unit[k] for k in ('text', 'caption', 'seed', 'num_steps')}}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def cache_path(unit_id, cache_key):
    """区切り文字・Unicode・ID衝突・パストラバーサルを避けたファイル名。"""
    digest = hashlib.sha256((unit_id + ':' + cache_key).encode('utf-8')).hexdigest()
    return SEG_DIR / f'unit_{digest}.wav'

def _load_runtime_unlocked(model_id=MODEL_ID):
    if _RUNTIME['instance'] is not None and _RUNTIME['model'] == model_id:
        return _RUNTIME['instance']
    import sys
    sys.path.insert(0, str(IRODORI_DIR))
    from huggingface_hub import hf_hub_download
    from irodori_tts.inference_runtime import InferenceRuntime, RuntimeKey
    ckpt = hf_hub_download(model_id, 'model.safetensors', revision=MODEL_REVISION)
    codec = hf_hub_download(CODEC_ID, 'weights.pth', revision=CODEC_REVISION)
    runtime = InferenceRuntime.from_key(RuntimeKey(checkpoint=ckpt, codec_repo=codec,
                                                   model_device='cuda', codec_device='cuda', model_precision='fp32', codec_precision='fp32'))
    _RUNTIME['instance'], _RUNTIME['model'] = runtime, model_id
    return runtime

import threading
_RUNTIME_LOCK=threading.Lock()
def load_runtime(model_id=MODEL_ID):
    with _RUNTIME_LOCK:
        return _load_runtime_unlocked(model_id)

def synthesize(unit, runtime):
    from irodori_tts.inference_runtime import SamplingRequest
    result = runtime.synthesize(SamplingRequest(text=unit['text'], caption=unit['caption'], ref_wav=unit['ref_wav'],
                                                max_ref_seconds=30, no_ref=False, num_steps=unit['num_steps'], seed=unit['seed']))
    audio = result.audios[0].detach().cpu().float().view(-1).numpy()
    if not audio.size or not np.isfinite(audio).all(): raise RuntimeError('空または異常な音声が生成されました')
    return audio, int(result.sample_rate)

def generate_all(plan, model_id=MODEL_ID, progress=None):
    runtime = load_runtime(model_id)
    files = []
    for n, unit in enumerate(plan, 1):
        key = request_key(unit, model_id)
        wav = cache_path(unit['unit_id'], key)
        meta = wav.with_suffix('.json')
        if wav.exists() and meta.exists() and json.loads(meta.read_text(encoding='utf-8')).get('key') == key:
            if progress: progress(f'[{n}/{len(plan)}] {unit["unit_id"]} は生成済みのため再利用')
        else:
            if progress: progress(f'[{n}/{len(plan)}] {unit["unit_id"]} を生成中…')
            audio, sr = synthesize(unit, runtime)
            sf.write(str(wav), audio, sr, subtype='PCM_16')
            meta.write_text(json.dumps({'key':key, 'unit_id':unit['unit_id'], 'segment_id':unit['segment_id']},ensure_ascii=False), encoding='utf-8')
        files.append((unit, wav))
    return files

def concat(files):
    if not files: raise ValueError('連結する音声がありません')
    chunks, sr_all = [], None
    for unit, wav in files:
        audio, sr = sf.read(str(wav), dtype='float32')
        if sr_all is None: sr_all = sr
        if sr != sr_all: raise ValueError('サンプルレートが混在しています')
        chunks.append(audio)
        pause = unit.get('pause_after_ms', 0)
        if pause and (unit, wav) != files[-1]:
            chunks.append(np.zeros(int(sr * pause / 1000), dtype='float32'))
    out = session_out_path('narration.wav')
    sf.write(str(out), np.concatenate(chunks), sr_all, subtype='PCM_16')
    return out

print('合成ロジックを読み込みました')

## 4. 画面（Web UI）

実行すると Gradio の画面が起動し、リンクが表示されます。画面の流れに沿って操作してください。
モデルの読み込みは、試作または本編の生成時に自動で行われます（初回は数分かかります）。

In [ ]:
print('Irodori-TTSモデルをGPUへ読み込み中です。完了後にWebUIを公開します。')
load_runtime()
print('モデルの準備が完了しました。WebUIを構築します。')
import sys, types
_module = types.ModuleType('notebook_v2')
sys.modules['notebook_v2'] = _module
exec('"""Koebuta v2 prototype: CPU-testable storage and candidate transactions.\n\nScenario parsing and generation plans are supplied by the existing P0 adapter.\nAll writes belong to a separate workspace, never the source project.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport stat\nimport tempfile\nimport threading\nimport uuid\nimport wave\nimport zipfile\n\n\ndef reference_choices(project, reading):\n    """Choose normal 20–100-character speech; no suitable line means no-ref."""\n    groups = {}\n    for segment in project[\'segments\']:\n        if segment.get(\'segment_type\') == \'separator\':\n            continue\n        key = \'shared\' if project.get(\'reading\') == \'shared\' else segment[\'role\']\n        groups.setdefault(key, []).append(segment)\n    result = {}\n    for key, segments in groups.items():\n        suitable = []\n        for segment in segments:\n            text = segment.get(\'tts_text\') or reading(segment[\'text\'], segment.get(\'text_format\', \'none\'))[0]\n            if (20 <= len(text.strip()) <= 100 and segment.get(\'emotion\', \'neutral\') == \'neutral\'\n                    and not any(c in text for c in \'!！?？…\') and len(set(text.strip())) >= 8):\n                suitable.append((abs(len(text) - 60), segment[\'id\'], text))\n        result[key] = min(suitable)[1:] if len(segments) >= 2 and suitable else None\n    return result\n\n\ndef atomic_bytes(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(dir=path.parent, prefix=\'.pending-\')\n    try:\n        with os.fdopen(fd, \'wb\') as stream:\n            stream.write(data)\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef unpack_project(archive, destination, max_bytes=512 * 1024 * 1024,\n                   max_entries=4096):\n    """Validate the complete central directory before extracting any member."""\n    destination = Path(destination).resolve()\n    if destination.exists():\n        raise ValueError(\'展開先には新しいフォルダを指定してください\')\n    with zipfile.ZipFile(archive) as source:\n        entries = source.infolist()\n        if len(entries) > max_entries or sum(x.file_size for x in entries) > max_bytes:\n            raise ValueError(\'ZIPの容量または項目数が上限を超えています\')\n        seen = set()\n        for entry in entries:\n            name = entry.filename.rstrip(\'/\')\n            parts = PurePosixPath(name).parts\n            mode = entry.external_attr >> 16\n            if (not name or \'\\\\\' in name or \':\' in name or \'\\x00\' in name\n                    or name.startswith(\'/\') or any(p in (\'\', \'.\', \'..\') for p in name.split(\'/\'))\n                    or stat.S_ISLNK(mode) or (stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR))\n                    or entry.flag_bits & 1 or name.casefold() in seen\n                    or (entry.file_size > 1024 * 1024 and entry.file_size > max(1, entry.compress_size) * 200)):\n                raise ValueError(\'安全でないZIP項目です: \' + name)\n            seen.add(name.casefold())\n        destination.mkdir(parents=True)\n        for entry in entries:\n            target = destination.joinpath(*PurePosixPath(entry.filename).parts)\n            if entry.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                atomic_bytes(target, source.read(entry))\n    return find_project(destination)\n\n\ndef find_project(folder):\n    root = Path(folder).resolve(strict=True)\n    candidates = [root / \'scenario.json\']\n    candidates += [p / \'scenario.json\' for p in root.iterdir() if p.is_dir() and not p.is_symlink()]\n    found = [p for p in candidates if p.is_file() and not p.is_symlink()]\n    if len(found) != 1:\n        raise ValueError(\'scenario.jsonを作品直下に一つ配置してください\')\n    return found[0].parent\n\n\ndef load_project(folder, normalize, validate, settings, resolve_wavs):\n    root = find_project(folder)\n    path = root / \'scenario.json\'\n    if path.stat().st_size > 16 * 1024 * 1024:\n        raise ValueError(\'シナリオJSONが16MiBを超えています\')\n    raw = path.read_bytes()\n    data = json.loads(raw)\n    if not isinstance(data, dict):\n        raise ValueError(\'JSONのルートはオブジェクトが必要です\')\n    project = normalize(data)\n    validate(project)\n    config = settings(project)\n    voices, warnings = resolve_wavs(project, root)\n    return project, config, voices, warnings, hashlib.sha256(raw).hexdigest()\n\n\ndef valid_wav(path):\n    try:\n        with wave.open(str(path), \'rb\') as audio:\n            return audio.getnframes() > 0 and len(audio.readframes(audio.getnframes())) == audio.getnframes() * audio.getnchannels() * audio.getsampwidth()\n    except (OSError, EOFError, wave.Error):\n        return False\n\n\nclass CandidateStore:\n    """Latest candidate per unit; accepted immutable blobs survive replacement.\n\n    The synthesizer writes a WAV to its supplied path. A group is published only\n    after every synthesis succeeds and every WAV is validated.\n    """\n    def __init__(self, workspace, identity):\n        self.root = Path(workspace)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.identity = dict(identity)\n        self.accepted, self.candidates, self.cache, self.references = {}, {}, {}, {}\n        self.session_settings = {}\n        self.lock = threading.RLock()\n\n    def key(self, unit):\n        payload = {k: v for k, v in unit.items()\n                   if k not in (\'unit_id\', \'segment_id\', \'pause_after_ms\', \'ref_wav\')}\n        payload[\'reference_sha256\'] = hashlib.sha256(Path(unit[\'ref_wav\']).read_bytes()).hexdigest() if unit.get(\'ref_wav\') else None\n        payload[\'identity\'] = self.identity\n        return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n    def regenerate(self, units, synthesize, force=True, progress=None):\n        with self.lock:\n            units = list(units)\n            staged = {}\n            for index, unit in enumerate(units):\n                if progress:\n                    progress(index / max(1, len(units)), desc=f"本文音声を確認中 {index + 1}/{len(units)}: {unit[\'unit_id\']}")\n                key = self.key(unit)\n                record = self.cache.get(key)\n                if force or not record or not self._valid(record):\n                    if progress:\n                        progress(index / max(1, len(units)), desc=f"本文音声を生成中 {index + 1}/{len(units)}: {unit[\'unit_id\']}")\n                    path = self.root / (uuid.uuid4().hex + \'.wav\')\n                    try:\n                        synthesize(unit, path)\n                        if not valid_wav(path):\n                            raise ValueError(\'生成WAVが破損しています\')\n                        record = {\'file\': path.name, \'key\': key,\n                                  \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest(),\n                                  \'unit\': {k: v for k, v in unit.items() if k != \'ref_wav\'},\n                                  \'identity\': self.identity}\n                    except Exception:\n                        path.unlink(missing_ok=True)\n                        raise\n                else:\n                    if progress:\n                        progress(index / max(1, len(units)), desc=f"キャッシュを再利用 {index + 1}/{len(units)}: {unit[\'unit_id\']}")\n                staged[unit[\'unit_id\']] = record\n                if progress:\n                    progress((index + 1) / len(units), desc=f"完了 {index + 1}/{len(units)}: {unit[\'unit_id\']}")\n            updated = {**self.candidates, **staged}\n            self._save(self.accepted, updated)\n            self.candidates = updated\n            self.cache.update({r[\'key\']: r for r in staged.values()})\n            return staged\n\n    def _valid(self, record):\n        name = record.get(\'file\', \'\')\n        if not name or Path(name).name != name:\n            return False\n        path = self.root / name\n        return valid_wav(path) and hashlib.sha256(path.read_bytes()).hexdigest() == record.get(\'sha256\')\n\n    def adopt(self, unit_ids):\n        with self.lock:\n            ids = list(unit_ids)\n            if not ids or any(i not in self.candidates or not self._valid(self.candidates[i]) for i in ids):\n                raise ValueError(\'全片の有効な候補が必要です\')\n            accepted = {**self.accepted, **{i: self.candidates[i] for i in ids}}\n            candidates = {i: r for i, r in self.candidates.items() if i not in ids}\n            self._save(accepted, candidates)\n            self.accepted, self.candidates = accepted, candidates\n\n    def discard(self, unit_ids):\n        with self.lock:\n            candidates = {i: r for i, r in self.candidates.items() if i not in unit_ids}\n            self._save(self.accepted, candidates)\n            self.candidates = candidates\n\n    def _save(self, accepted, candidates):\n        cache = {**self.cache, **{r[\'key\']: r for group in (accepted, candidates) for r in group.values()}}\n        atomic_bytes(self.root / \'state.json\', json.dumps({\n            \'version\': 2, \'identity\': self.identity, \'accepted\': accepted,\n            \'candidates\': candidates, \'cache\': cache, \'references\': self.references,\n            \'session_settings\': self.session_settings}, ensure_ascii=False, allow_nan=False).encode())\n\n    def restore(self):\n        with self.lock:\n            state = json.loads((self.root / \'state.json\').read_bytes())\n            if state.get(\'version\') != 2 or state.get(\'identity\') != self.identity:\n                raise ValueError(\'保存状態のモデル識別情報が一致しません\')\n            groups = [state[\'accepted\'], state[\'candidates\']]\n            if any(not self._valid(r) for group in groups for r in group.values()):\n                raise ValueError(\'保存状態の音声が欠落または破損しています\')\n            self.accepted, self.candidates = groups\n            self.session_settings = state.get(\'session_settings\', {})\n            self.references = {k: r for k, r in state.get(\'references\', {}).items() if self._valid(r)}\n            self.cache = {k: r for k, r in state.get(\'cache\', {}).items() if self._valid(r)}\n            self.cache.update({r[\'key\']: r for group in groups for r in group.values()})\n\n    def persist(self, destination):\n        """Copy audio first, publish the manifest last; local results stay intact."""\n        try:\n            destination = Path(destination)\n            if destination.resolve() == self.root.resolve():\n                return None\n            for group in (self.accepted, self.candidates, self.cache, self.references):\n                for record in group.values():\n                    if not self._valid(record):\n                        raise ValueError(\'保存対象WAVが破損しています\')\n                    atomic_bytes(destination / record[\'file\'], (self.root / record[\'file\']).read_bytes())\n            atomic_bytes(destination / \'state.json\', (self.root / \'state.json\').read_bytes())\n            return None\n        except Exception as exc:\n            return \'Drive保存に失敗しました。一時領域の結果は保持しています: \' + str(exc)\n', _module.__dict__)
_module = types.ModuleType('notebook_v2_ui')
sys.modules['notebook_v2_ui'] = _module
exec('"""Generation-only Gradio prototype, executed in the P0 Notebook namespace."""\nimport functools\nimport tempfile\nimport uuid\nimport zipfile\nimport gradio as gr\n\n\ndef create_v2_ui(api):\n    from notebook_v2 import CandidateStore, load_project, unpack_project, atomic_bytes, valid_wav\n    import numpy as np\n    import soundfile as sf\n    from pathlib import Path\n\n    def session():\n        root = Path(tempfile.mkdtemp(prefix=\'koebuta-v2-\'))\n        return {\'root\': root, \'project\': None, \'voices\': {}, \'store\': None, \'plan\': [], \'sample\': None}\n\n    def guarded(fn):\n        @functools.wraps(fn)\n        def call(state, *args, **kwargs):\n            state = state or session()\n            token = api[\'_CTX\'].set(api[\'new_session\']())\n            try:\n                result = fn(state, *args, **kwargs)\n                if not isinstance(result, tuple):\n                    result = (result,)\n                return (*result, state)\n            except Exception as exc:\n                raise gr.Error(str(exc)) from exc\n            finally:\n                api[\'_CTX\'].reset(token)\n        return call\n\n    def plan(state, seed, steps, acting, emotion, narrator, pause, prepare=False, selected_role=None, progress=None):\n        if state[\'project\'] is None:\n            raise ValueError(\'作品を読み込んでください\')\n        from notebook_v2 import reference_choices\n        import hashlib\n        choices = reference_choices(state[\'project\'], api[\'reading\'])\n        reference_keys = [key for key, choice in choices.items() if choice and key not in state[\'voices\']\n                          and (selected_role is None or key == selected_role)\n                          and not (state[\'store\'].references.get(key) and state[\'store\']._valid(state[\'store\'].references[key]))]\n        if prepare and progress:\n            progress(0.05, desc=\'参照音声を確認中\')\n        voices = dict(state[\'voices\'])\n        for key, choice in choices.items():\n            if key in voices:\n                continue\n            record = state[\'store\'].references.get(key)\n            if record and state[\'store\']._valid(record):\n                voices[key] = str(state[\'store\'].root / record[\'file\'])\n            elif prepare and choice and (selected_role is None or key == selected_role):\n                role = next(r for r in state[\'project\'][\'roles\'] if r[\'id\'] == next(s[\'role\'] for s in state[\'project\'][\'segments\'] if s[\'id\'] == choice[0]))\n                unit = dict(text=choice[1], caption=(state[\'project\'].get(\'baseCaption\') if key == \'shared\' else role.get(\'caption\')) or None,\n                            ref_wav=None, no_ref=True, seed=int(seed), num_steps=int(steps))\n                path = state[\'store\'].root / (uuid.uuid4().hex + \'.wav\')\n                if progress:\n                    index = reference_keys.index(key)\n                    progress(0.05 + 0.2 * index / max(1, len(reference_keys)),\n                             desc=f"自動参照音声を生成中 {index + 1}/{len(reference_keys)}: {role[\'name\']} ({choice[0]})")\n                synthesis(unit, path)\n                if not valid_wav(path):\n                    raise ValueError(\'自動参照音声が不正です\')\n                record = dict(file=path.name, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                              source_segment=choice[0], text=choice[1], seed=int(seed), steps=int(steps))\n                state[\'store\'].references[key] = record\n                state[\'store\']._save(state[\'store\'].accepted, state[\'store\'].candidates)\n                voices[key] = str(path)\n                if progress:\n                    progress(0.05 + 0.2 * (reference_keys.index(key) + 1) / max(1, len(reference_keys)),\n                             desc=f"自動参照音声が完了: {role[\'name\']}")\n            else:\n                voices[key] = None\n        units = api[\'build_plan\'](state[\'project\'], voices, use_acting=acting,\n            use_emotion=emotion, narrator_emotion=narrator, seed=int(seed),\n            steps=int(steps), global_pause_ms=int(pause))\n        roles = {r[\'id\']: r for r in state[\'project\'][\'roles\']}\n        segments = {s[\'id\']: s for s in state[\'project\'][\'segments\']}\n        for unit in units:\n            key = \'shared\' if state[\'project\'].get(\'reading\') == \'shared\' else segments[unit[\'segment_id\']][\'role\']\n            unit[\'ref_wav\'] = voices[key]\n            unit[\'no_ref\'] = voices[key] is None\n            if unit[\'no_ref\']:\n                role_caption = roles[segments[unit[\'segment_id\']][\'role\']].get(\'caption\', \'\')\n                unit[\'caption\'] = \'\\n\'.join(x for x in [role_caption, unit[\'caption\']] if x) or None\n        state[\'plan\'] = units\n        state[\'settings\'] = [int(seed), int(steps), acting, emotion, narrator, int(pause)]\n        return units\n\n    def summary(state):\n        segments = {s[\'id\']: s for s in state[\'project\'][\'segments\']}\n        return [[u[\'unit_id\'], segments[u[\'segment_id\']].get(\'role\'),\n                 segments[u[\'segment_id\']][\'text\'][:45], u[\'segment_id\'],\n                 \'候補あり\' if u[\'unit_id\'] in state[\'store\'].candidates else\n                 (\'採用済み\' if u[\'unit_id\'] in state[\'store\'].accepted else \'未生成\')] for u in state[\'plan\']]\n\n    def speaker_choices(state):\n        project = state[\'project\']\n        people = [(\'shared\', \'共通の声\')] if project.get(\'reading\') == \'shared\' else [(r[\'id\'], r[\'name\']) for r in project[\'roles\']]\n        result = []\n        for key, name in people:\n            record = state[\'store\'].references.get(key)\n            configured = bool(state[\'voices\'].get(key)) or bool(record and state[\'store\']._valid(record))\n            result.append((f"{key} · {name} · 声設定 {\'有\' if configured else \'無\'}", key))\n        return result\n\n    @guarded\n    def role_info(state, role):\n        if not state.get(\'project\') or not role:\n            return \'\', gr.update(choices=[], value=None), None\n        person = next((r for r in state[\'project\'][\'roles\'] if r[\'id\'] == role), None)\n        if state.get(\'sample_role\') != role:\n            state[\'sample\'] = None\n        state[\'active_role\'] = role\n        record = state[\'store\'].references.get(role)\n        path = state[\'voices\'].get(role)\n        if not path and record and state[\'store\']._valid(record):\n            path = str(state[\'store\'].root / record[\'file\'])\n        return (person.get(\'caption\', \'\') if person else state[\'project\'].get(\'baseCaption\', \'\'),\n                gr.update(choices=speaker_choices(state), value=role), path)\n\n    @guarded\n    def edit_caption(state, role, value):\n        if state.get(\'project\'):\n            if role == \'shared\':\n                state[\'project\'][\'baseCaption\'] = value\n            else:\n                person = next(r for r in state[\'project\'][\'roles\'] if r[\'id\'] == role)\n                person[\'caption\'] = value\n            state[\'store\'].references.pop(role, None)\n        return ()\n\n    def random_seed():\n        import secrets\n        return secrets.randbelow(2147483648)\n\n    @guarded\n    def details(state, selected):\n        unit = next((u for u in state[\'plan\'] if u[\'unit_id\'] == selected), None)\n        if unit is None:\n            return \'\', None, None\n        segment = next(s for s in state[\'project\'][\'segments\'] if s[\'id\'] == unit[\'segment_id\'])\n        store = state[\'store\']\n        def audio(records):\n            record = records.get(selected)\n            return str(store.root / record[\'file\']) if record and store._valid(record) else None\n        return (f"{selected}\\n話者: {segment[\'role\']} / 感情: {segment[\'emotion\']} / 休止: {unit[\'pause_after_ms\']}ms\\n"\n                f"読み上げ文: {unit[\'text\']}\\n演技・声質: {unit[\'caption\'] or \'指定なし\'}",\n                audio(store.accepted), audio(store.candidates))\n\n    @guarded\n    def load(state, archive, folder):\n        if archive and Path(archive).suffix.lower() == \'.json\':\n            source = state[\'root\'] / uuid.uuid4().hex\n            path = Path(archive)\n            if path.stat().st_size > 16 * 1024 * 1024:\n                raise ValueError(\'シナリオJSONが16MiBを超えています\')\n            atomic_bytes(source / \'scenario.json\', path.read_bytes())\n        else:\n            source = unpack_project(archive, state[\'root\'] / uuid.uuid4().hex) if archive else folder\n        project, config, voices, warnings, digest = load_project(source,\n            api[\'normalize_scenario\'], api[\'require_ready\'], api[\'notebook_settings\'], api[\'resolve_project_wavs\'])\n        from notebook_v2 import find_project\n        state.update(project=project, voices=voices, plan=[], source=find_project(source))\n        identity = {k: api.get(k, \'unknown\') for k in\n                    (\'MODEL_ID\', \'MODEL_REVISION\', \'CODEC_ID\', \'CODEC_REVISION\', \'IRODORI_RESOLVED_COMMIT\')}\n        identity[\'scenario_sha256\'] = digest\n        state[\'store\'] = CandidateStore(state[\'root\'] / uuid.uuid4().hex, identity)\n        roles = [\'shared\'] if project.get(\'reading\') == \'shared\' else [r[\'id\'] for r in project[\'roles\']]\n        role_choices = speaker_choices(state)\n        units = plan(state, config.get(\'seed\', 314), config.get(\'steps\', 40), config.get(\'use_acting\', False),\n                     config.get(\'use_emotion\', True), config.get(\'narrator_emotion\', False), config.get(\'default_pause_ms\', 280))\n        return (f"発話数: {sum(s.get(\'segment_type\') != \'separator\' for s in project[\'segments\'])} / 参照音声数: {len(voices)}\\n" + \'\\n\'.join(warnings),\n                gr.update(choices=role_choices, value=roles[0] if roles else None),\n                config.get(\'seed\', 314), config.get(\'steps\', 40), config.get(\'use_acting\', False),\n                config.get(\'use_emotion\', True), config.get(\'narrator_emotion\', False), config.get(\'default_pause_ms\', 280),\n                str(voices), summary(state), gr.update(choices=[u[\'unit_id\'] for u in units], value=units[0][\'unit_id\']))\n\n    def synthesis(unit, path):\n        runtime = api[\'load_runtime\']()\n        if unit.get(\'no_ref\'):\n            from irodori_tts.inference_runtime import SamplingRequest\n            result = runtime.synthesize(SamplingRequest(text=unit[\'text\'], caption=unit[\'caption\'],\n                    no_ref=True, num_steps=unit[\'num_steps\'], seed=unit[\'seed\']))\n            audio, sr = result.audios[0].detach().cpu().float().view(-1).numpy(), int(result.sample_rate)\n        else:\n            audio, sr = api[\'synthesize\'](unit, runtime)\n        sf.write(str(path), audio, sr, subtype=\'PCM_16\')\n\n    @guarded\n    def inspect(state, *settings):\n        units = plan(state, *settings)\n        return summary(state), gr.update(choices=[u[\'unit_id\'] for u in units], value=units[0][\'unit_id\']), f\'生成単位数: {len(units)}\\n\' + \'\\n\'.join(api[\'STATE\'][\'plan_warnings\'])\n\n    def targets(state, selected, group):\n        unit = next((u for u in state[\'plan\'] if u[\'unit_id\'] == selected), None)\n        if unit is None:\n            raise ValueError(\'生成計画を更新して対象を選んでください\')\n        return [u for u in state[\'plan\'] if u[\'segment_id\'] == unit[\'segment_id\']] if group else [unit]\n\n    @guarded\n    def regenerate(state, selected, group, seed, steps, acting, emotion, narrator, pause, progress=gr.Progress()):\n        settings = (seed, steps, acting, emotion, narrator, pause)\n        progress(0, desc=\'再生成の対象を確認中\')\n        target = targets(state, selected, group)[0]\n        segment = next(s for s in state[\'project\'][\'segments\'] if s[\'id\'] == target[\'segment_id\'])\n        selected_role = \'shared\' if state[\'project\'].get(\'reading\') == \'shared\' else segment[\'role\']\n        plan(state, *settings, prepare=True, selected_role=selected_role, progress=progress)\n        state[\'store\'].regenerate(targets(state, selected, group), synthesis,\n            progress=lambda value, **kw: progress(0.25 + 0.7 * value, **kw))\n        progress(1, desc=\'候補の生成・保存が完了しました\')\n        return summary(state), \'候補を更新しました。試聴後に採用してください\'\n\n    @guarded\n    def select(state, selected):\n        store = state[\'store\']\n        def audio(group):\n            record = group.get(selected)\n            return str(store.root / record[\'file\']) if record and store._valid(record) else None\n        return audio(store.accepted), audio(store.candidates)\n\n    @guarded\n    def adopt(state, selected, group, *settings):\n        if settings:\n            state[\'settings\'] = list(settings)\n        if state.get(\'settings\'):\n            plan(state, *state[\'settings\'])\n        for unit in targets(state, selected, group):\n            record = state[\'store\'].candidates.get(unit[\'unit_id\'])\n            if record and record[\'key\'] != state[\'store\'].key(unit):\n                raise ValueError(\'現在の条件と異なる候補です。設定を戻すか再生成してください\')\n        state[\'store\'].adopt([u[\'unit_id\'] for u in targets(state, selected, group)])\n        return summary(state), \'採用しました。完成WAVは再結合で更新できます\'\n\n    @guarded\n    def discard(state, selected, group):\n        state[\'store\'].discard([u[\'unit_id\'] for u in targets(state, selected, group)])\n        return summary(state), \'候補を破棄しました\'\n\n    @guarded\n    def assign(state, role, upload):\n        if not upload or not valid_wav(upload):\n            raise ValueError(\'有効なPCM WAVを選んでください\')\n        target = state[\'root\'] / (uuid.uuid4().hex + \'.wav\')\n        atomic_bytes(target, Path(upload).read_bytes())\n        state[\'voices\'][role] = str(target)\n        return str(state[\'voices\']), str(target)\n\n    @guarded\n    def sample(state, text, caption, seed, steps, progress=gr.Progress()):\n        unit = dict(text=text, caption=caption or None, ref_wav=None, seed=int(seed), num_steps=int(steps))\n        # Voice design is reference-free, matching the v1 runtime request.\n        progress(0, desc=\'モデル準備中（初回は取得・GPU読み込みに時間がかかります）\')\n        runtime = api[\'load_runtime\']()\n        from irodori_tts.inference_runtime import SamplingRequest\n        progress(0.5, desc=\'声の試作を生成中\')\n        result = runtime.synthesize(SamplingRequest(text=text, caption=caption or None,\n                    no_ref=True, num_steps=int(steps), seed=int(seed)))\n        target = state[\'root\'] / (uuid.uuid4().hex + \'.wav\')\n        sf.write(str(target), result.audios[0].detach().cpu().float().view(-1).numpy(), int(result.sample_rate), subtype=\'PCM_16\')\n        if not valid_wav(target):\n            raise ValueError(\'試作音声が不正です\')\n        state[\'sample\'] = str(target)\n        state[\'sample_role\'] = state.get(\'active_role\')\n        progress(1, desc=\'声の試作が完了しました\')\n        return str(target)\n\n    @guarded\n    def adopt_sample(state, role):\n        if not state[\'sample\']:\n            raise ValueError(\'声を試作してください\')\n        if state.get(\'sample_role\') != role:\n            raise ValueError(\'別の人物の試作音声です。この人物を選択して試作してください\')\n        state[\'voices\'][role] = state[\'sample\']\n        return str(state[\'voices\'])\n\n    @guarded\n    def generate(state, seed, steps, acting, emotion, narrator, pause, progress=gr.Progress()):\n        settings = (seed, steps, acting, emotion, narrator, pause)\n        progress(0, desc=\'全編生成の計画を確認中\')\n        units = plan(state, *settings, prepare=True, progress=progress)\n        missing = [u for u in units if u[\'unit_id\'] not in state[\'store\'].accepted\n                   or not state[\'store\']._valid(state[\'store\'].accepted[u[\'unit_id\']])\n                   or state[\'store\'].accepted[u[\'unit_id\']][\'key\'] != state[\'store\'].key(u)]\n        if missing:\n            pending = [u[\'unit_id\'] for u in missing if u[\'unit_id\'] in state[\'store\'].candidates]\n            if pending:\n                raise ValueError(\'試聴待ち候補を先に採用または破棄してください: \' + \'、\'.join(pending))\n            state[\'store\'].regenerate(missing, synthesis, force=False,\n                progress=lambda value, **kw: progress(0.25 + 0.7 * value, **kw))\n            progress(0.96, desc=\'生成結果を採用・保存中\')\n            state[\'store\'].adopt([u[\'unit_id\'] for u in missing])\n        progress(1, desc=\'全編の音声準備が完了しました\')\n        return summary(state), \'全編の採用済み音声を準備しました。再結合してください\'\n\n    @guarded\n    def output(state, destination, seed, steps, acting, emotion, narrator, pause, progress=gr.Progress()):\n        settings = (seed, steps, acting, emotion, narrator, pause)\n        progress(0, desc=\'採用音声の整合性を確認中\')\n        units = plan(state, *settings)\n        store = state[\'store\']\n        chunks, rate = [], None\n        zip_path = state[\'root\'] / (uuid.uuid4().hex + \'.zip\')\n        # Validate all accepted inputs before publishing any output.\n        for unit in units:\n            record = store.accepted.get(unit[\'unit_id\'])\n            if not record or not store._valid(record) or record[\'key\'] != store.key(unit):\n                raise ValueError(unit[\'unit_id\'] + \': 現在の条件に一致する採用音声が必要です\')\n        with zipfile.ZipFile(zip_path, \'w\', zipfile.ZIP_DEFLATED) as archive:\n            for index, unit in enumerate(units):\n                progress(index / len(units) * 0.8, desc=f"結合・ZIP作成 {index + 1}/{len(units)}")\n                path = store.root / store.accepted[unit[\'unit_id\']][\'file\']\n                audio, sr = sf.read(str(path), dtype=\'float32\')\n                if audio.ndim != 1 or (rate is not None and sr != rate):\n                    raise ValueError(\'音声の形式が一致しません\')\n                rate = sr\n                chunks.append(audio)\n                if index < len(units) - 1:\n                    chunks.append(np.zeros(int(sr * unit[\'pause_after_ms\'] / 1000), dtype=\'float32\'))\n                archive.write(path, f\'{index + 1:06d}.wav\')\n        target = state[\'root\'] / (uuid.uuid4().hex + \'.wav\')\n        sf.write(str(target), np.concatenate(chunks), rate, subtype=\'PCM_16\')\n        message = \'完成WAVと個別音声ZIPを作成しました\'\n        if destination.strip():\n            progress(0.9, desc=\'Driveへ保存中（一時領域にも保持します）\')\n            resolved = Path(destination).resolve()\n            if resolved == state[\'source\'] or resolved.is_relative_to(state[\'source\']):\n                raise ValueError(\'元作品の保護のため、作品フォルダ外の保存先を指定してください。完成結果は一時領域に保持しています\')\n            saved_message = save_state(state, destination)[0]\n            warning = saved_message if \'失敗\' in saved_message else None\n            try:\n                atomic_bytes(Path(destination) / \'narration.wav\', target.read_bytes())\n                atomic_bytes(Path(destination) / \'individual.zip\', zip_path.read_bytes())\n            except Exception as exc:\n                warning = \'Drive出力保存失敗。一時領域の結果は保持しています: \' + str(exc)\n            message += \'\\n\' + (warning or \'Driveへ保存しました\')\n        progress(1, desc=\'完成WAV・ZIPの出力が完了しました\')\n        return str(target), str(target), str(zip_path), message\n\n    @guarded\n    def save_state(state, destination, *settings):\n        import hashlib\n        if not state.get(\'store\') or not destination.strip():\n            raise ValueError(\'作品と保存先を指定してください\')\n        resolved = Path(destination).resolve()\n        if resolved == state[\'source\'] or resolved.is_relative_to(state[\'source\']):\n            raise ValueError(\'元作品フォルダ外の保存先を指定してください\')\n        store = state[\'store\']\n        if settings:\n            plan(state, *settings)\n        manual = {}\n        for role_id, source in state[\'voices\'].items():\n            data = Path(source).read_bytes()\n            digest = hashlib.sha256(data).hexdigest()\n            name = \'reference_\' + digest + \'.wav\'\n            atomic_bytes(store.root / name, data)\n            store.references[\'manual:\' + role_id] = dict(file=name, sha256=digest)\n            manual[role_id] = name\n        store.session_settings = {\'captions\': {r[\'id\']: r.get(\'caption\', \'\') for r in state[\'project\'][\'roles\']},\n                                  \'baseCaption\': state[\'project\'].get(\'baseCaption\', \'\'), \'manual\': manual,\n                                  \'generation\': state.get(\'settings\', [314, 40, False, True, False, 280])}\n        store._save(store.accepted, store.candidates)\n        return store.persist(destination) or \'作業途中の候補・採用状態と声の設定を保存しました\'\n\n    @guarded\n    def restore(state, folder):\n        source = CandidateStore(folder, state[\'store\'].identity)\n        source.restore()\n        warning = source.persist(state[\'store\'].root)\n        if warning:\n            raise ValueError(warning)\n        state[\'store\'].restore()\n        saved = state[\'store\'].session_settings\n        for person in state[\'project\'][\'roles\']:\n            person[\'caption\'] = saved.get(\'captions\', {}).get(person[\'id\'], person.get(\'caption\', \'\'))\n        state[\'project\'][\'baseCaption\'] = saved.get(\'baseCaption\', state[\'project\'].get(\'baseCaption\', \'\'))\n        state[\'voices\'] = {key: str(state[\'store\'].root / name) for key, name in saved.get(\'manual\', {}).items()\n                           if Path(name).name == name and valid_wav(state[\'store\'].root / name)}\n        restored_settings = saved.get(\'generation\', [314, 40, False, True, False, 280])\n        plan(state, *restored_settings)\n        return (\'保存状態と生成条件を復元しました\', *restored_settings, str(state[\'voices\']))\n\n    with gr.Blocks() as demo:\n        state = gr.State(None)\n        gr.Markdown(\'# Koebuta Notebook v2 試作\\nシナリオJSONは読み取り専用です。編集はHTML版で行ってください。\')\n        with gr.Row():\n          with gr.Column(scale=1, min_width=400):\n            gr.Markdown(\'## 1. 作品の読み込み\')\n            archive = gr.File(label=\'作品ZIP / シナリオJSON（アップロードを優先）\', type=\'filepath\', file_types=[\'.zip\', \'.json\'])\n            gr.Markdown(\'JSON単体では参照WAVは付属しません。声は自動生成または手動で割り当てられます。\')\n            folder = gr.Textbox(label=\'マウント済みDrive作品フォルダ\')\n            load_button = gr.Button(\'作品を読み込む\')\n            validation = gr.Textbox(label=\'構造検証\', interactive=False, lines=2)\n          with gr.Column(scale=1, min_width=400):\n            gr.Markdown(\'## 2. 声の設定\')\n            role = gr.Dropdown(label=\'人物一覧（話者ID・名前・声設定）\')\n            voices = gr.Textbox(label=\'参照WAV割り当て\', interactive=False)\n            with gr.Accordion(\'参照WAVの割り当て・試聴\', open=False):\n                upload = gr.File(label=\'参照WAV\', type=\'filepath\')\n                assign_button = gr.Button(\'参照WAVを割り当て\')\n                reference_audio = gr.Audio(label=\'参照音声\')\n            gr.Markdown(\'WAV未設定でも生成できます。適切な台詞から声を自動作成し、候補がなければ参照なしで生成します。\')\n            with gr.Accordion(\'声の試作\', open=False):\n                sample_text = gr.Textbox(label=\'試作の読み上げ文\', value=\'こんにちは。今日はどのようなお話をいたしましょうか。\')\n                caption = gr.Textbox(label=\'試作する声の説明\')\n                sample_button = gr.Button(\'声を試作\')\n                sample_audio = gr.Audio(label=\'試作音声\')\n                adopt_sample_button = gr.Button(\'試作を話者の参照音声に採用\')\n        with gr.Group():\n            gr.Markdown(\'## 3. 生成・試聴・採用\')\n            with gr.Row():\n                with gr.Column(scale=1, min_width=250):\n                    with gr.Group(elem_id=\'seed-field\'):\n                        seed = gr.Number(label=\'Seed\', value=314, precision=0)\n                        random_button = gr.Button(\'🎲\', size=\'sm\', scale=0, min_width=32, elem_id=\'seed-random\')\n                steps = gr.Slider(1, 200, value=40, step=1, label=\'Steps\')\n                pause = gr.Slider(0, 60000, value=280, step=1, label=\'共通休止時間 ms\')\n            with gr.Row():\n                acting = gr.Checkbox(label=\'演技を反映\')\n                emotion = gr.Checkbox(label=\'感情を反映\', value=True)\n                narrator = gr.Checkbox(label=\'地の文にも感情を反映\')\n            settings = [seed, steps, acting, emotion, narrator, pause]\n            inspect_button = gr.Button(\'生成計画を更新\')\n            with gr.Row():\n              with gr.Column(scale=1, min_width=400):\n                table = gr.Dataframe(headers=[\'生成単位\', \'話者\', \'原文（抜粋）\', \'発話\', \'状態\'], interactive=False)\n              with gr.Column(scale=1, min_width=400):\n                selected = gr.Dropdown(label=\'選択中のTTS生成単位\')\n                detail = gr.Textbox(label=\'選択対象の詳細（読み取り専用）\', interactive=False, lines=4)\n                group = gr.Checkbox(label=\'同じ発話の全片をまとめて操作\', value=False)\n                with gr.Row():\n                    accepted_audio = gr.Audio(label=\'採用済み音声\')\n                    candidate_audio = gr.Audio(label=\'最新候補\')\n                with gr.Row():\n                    regenerate_button = gr.Button(\'新しい候補を生成\', variant=\'primary\')\n                with gr.Row():\n                    adopt_button = gr.Button(\'候補を採用\', variant=\'primary\')\n                    discard_button = gr.Button(\'候補を破棄\')\n                gr.Markdown(\'未採用候補は、各生成単位につき最新1件です。\')\n        with gr.Group():\n            gr.Markdown(\'## 4. 出力\')\n            with gr.Row():\n              with gr.Column(scale=3):\n                destination = gr.Textbox(label=\'Drive保存先（空欄なら一時領域のみ）\')\n                with gr.Row():\n                    generate_button = gr.Button(\'全編生成\', variant=\'primary\')\n                    output_button = gr.Button(\'完成WAV再結合・出力\')\n                    restore_button = gr.Button(\'保存状態を復元\')\n                    save_button = gr.Button(\'作業状態を保存\')\n                master = gr.Audio(label=\'完成WAV\')\n                with gr.Row():\n                    download = gr.File(label=\'完成WAV保存\')\n                    individual = gr.File(label=\'個別WAV ZIP\')\n              with gr.Column(scale=2):\n                status = gr.Textbox(label=\'処理ログ・進捗\', interactive=False, lines=8)\n        queue = dict(concurrency_id=\'koebuta-v2\', concurrency_limit=1)\n        random_button.click(random_seed, [], [seed], **queue)\n        save_button.click(save_state, [state, destination, *settings], [status, state], **queue)\n        role.input(role_info, [state, role], [caption, role, reference_audio, state], **queue)\n        caption.input(edit_caption, [state, role, caption], [state], **queue)\n        load_button.click(load, [state, archive, folder], [validation, role, *settings, voices, table, selected, state], **queue).then(\n            role_info, [state, role], [caption, role, reference_audio, state], **queue)\n        inspect_button.click(inspect, [state, *settings], [table, selected, status, state], **queue)\n        assign_button.click(assign, [state, role, upload], [voices, reference_audio, state], **queue).then(\n            role_info, [state, role], [caption, role, reference_audio, state], **queue)\n        sample_button.click(sample, [state, sample_text, caption, seed, steps], [sample_audio, state], **queue)\n        adopt_sample_button.click(adopt_sample, [state, role], [voices, state], **queue).then(\n            role_info, [state, role], [caption, role, reference_audio, state], **queue)\n        selected.change(details, [state, selected], [detail, accepted_audio, candidate_audio, state], **queue)\n        for button, fn, inputs in ((regenerate_button, regenerate, [state, selected, group, *settings]),\n                                   (adopt_button, adopt, [state, selected, group, *settings]),\n                                   (discard_button, discard, [state, selected, group])):\n            button.click(fn, inputs, [table, status, state], **queue).then(\n                details, [state, selected], [detail, accepted_audio, candidate_audio, state], **queue)\n        generate_button.click(generate, [state, *settings], [table, status, state], **queue).then(\n            role_info, [state, role], [caption, role, reference_audio, state], **queue).then(\n            details, [state, selected], [detail, accepted_audio, candidate_audio, state], **queue)\n        output_button.click(output, [state, destination, *settings], [master, download, individual, status, state], **queue)\n        restore_button.click(restore, [state, destination], [status, *settings, voices, state], **queue).then(\n            inspect, [state, *settings], [table, selected, status, state], **queue).then(\n            role_info, [state, role], [caption, role, reference_audio, state], **queue)\n    return demo\n', _module.__dict__)
from notebook_v2_ui import create_v2_ui
import secrets, os, gradio as gr
demo = create_v2_ui(globals())
ui_user = 'koebuta'
ui_password = secrets.token_urlsafe(16)
print('ユーザー:', ui_user, 'パスワード:', ui_password)
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, auth=(ui_user, ui_password), theme=gr.themes.Soft(primary_hue='purple'), css="#seed-field{position:relative}#seed-random{position:absolute!important;top:4px;right:8px;left:auto!important;z-index:5;min-height:22px!important;height:22px!important;padding:0!important;width:32px!important;min-width:32px!important;max-width:32px!important}", show_error=True)
